In [19]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('netflix_catalogue.csv')
print(f"Loaded: {len(df)} titles")
print(df['type'].value_counts())
print(df.head())

Loaded: 3000 titles
type
Movie      1974
TV Show    1026
Name: count, dtype: int64
      type  release_year  added_year             genre        country rating  \
0    Movie          2014        2016  Sci-Fi & Fantasy         France  PG-13   
1    Movie          2010        2014     Documentaries  United States  TV-MA   
2  TV Show          2011        2012     Kids & Family  United States  TV-14   
3    Movie          2016        2018             Anime          India     PG   
4    Movie          2014        2016     Kids & Family         Canada  TV-MA   

   duration  
0       157  
1       127  
2         6  
3       134  
4        77  


In [20]:
print("Genres:", df['genre'].value_counts().head(8))
print("\nCountries:", df['country'].value_counts().head(8))
print("\nRatings:", df['rating'].value_counts())

Genres: genre
Sports                244
Sci-Fi & Fantasy      213
Kids & Family         209
Crime                 206
Drama                 204
Horror                199
Action & Adventure    198
Thrillers             195
Name: count, dtype: int64

Countries: country
United States     932
India             337
United Kingdom    261
Japan             187
France            176
Canada            164
South Korea       151
Mexico            138
Name: count, dtype: int64

Ratings: rating
TV-MA    840
TV-14    733
PG-13    589
R        312
PG       196
TV-PG    128
G         92
TV-Y7     57
TV-G      53
Name: count, dtype: int64


In [21]:
import pandas as pd
import plotly.express as px

df["decade"] = (df["release_year"] // 10 * 10).astype(str) + "s"

target_ratings = ["TV-14", "TV-MA", "PG-13", "R", "PG"]
df_filtered = df[df["rating"].isin(target_ratings)]

heatmap_data = (
    df_filtered.groupby(["rating", "decade"])
    .size()
    .reset_index(name="count")
)

heatmap_pivot = heatmap_data.pivot(
    index="rating", columns="decade", values="count"
).fillna(0)

fig = px.imshow(
    heatmap_pivot,
    labels=dict(x="Decade", y="Content Rating", color="Title Count"),
    x=heatmap_pivot.columns,
    y=heatmap_pivot.index,
    color_continuous_scale="Blues",
    text_auto=True,
    title="<b>The Rise of Adult Streaming: TV-MA Content Dominates the 2010s and 2020s Catalog Expansion</b>",
)

fig.update_layout(
    title_font_size=16,
    xaxis_title="Decade",
    yaxis_title="Rating",
    coloraxis_showscale=True,
)

fig.show()

In [22]:
import pandas as pd
import plotly.graph_objects as go

df["added_year"] = pd.to_numeric(df["added_year"], errors="coerce")
df_movies = df[(df["type"] == "Movie") & (df["added_year"].between(2015, 2022))]

movie_growth = (
    df_movies.groupby("added_year").size().reset_index(name="additions")
)
all_years = pd.DataFrame({"added_year": range(2015, 2023)})
movie_growth = pd.merge(all_years, movie_growth, on="added_year", how="left").fillna(0)

years = movie_growth["added_year"].astype(int).astype(str).tolist()
additions = movie_growth["additions"].astype(int).tolist()

max_addition_idx = movie_growth["additions"].idxmax()
max_year = years[max_addition_idx]
max_value = additions[max_addition_idx]

cumulative_totals = []
current_sum = 0
for val in additions:
    current_sum += val
    cumulative_totals.append(current_sum)
cumulative_total_at_max = cumulative_totals[max_addition_idx]

x_labels = years + ["Total"]
measure_types = ["relative"] * len(years) + ["total"]
y_values = additions + [0]

fig = go.Figure(
    go.Waterfall(
        name="Movie Additions",
        orientation="v",
        measure=measure_types,
        x=x_labels,
        y=y_values,
        text=[f"+{v}" if v > 0 else str(v) for v in additions] + [str(sum(additions))],
        textposition="outside",
        cliponaxis=False,
        increasing=dict(marker=dict(color="green")),
        totals=dict(marker=dict(color="blue")),
    )
)

fig.add_annotation(
    x=max_year,
    y=cumulative_total_at_max,
    text=f"Peak Expansion Year<br>(+{max_value} Movies)",
    showarrow=True,
    arrowhead=2,
    arrowcolor="black",
    ax=0,
    ay=-60,
    font=dict(color="black", size=11),
    bgcolor="white",
    bordercolor="black",
    borderwidth=1,
)

fig.update_layout(
    title="<b>Aggressive Library Expansion: Movie Releases Accelerated Rapidly, Peaking in 2019 Before a Post-Pandemic Slowdown</b>",
    title_font_size=16,
    xaxis_title="Year Added to Netflix",
    yaxis_title="Number of Movies",
    xaxis=dict(
        type="category",
        tickmode="array",
        tickvals=x_labels,
        range=[-0.5, len(x_labels) - 0.5]
    ),
    yaxis=dict(
        range=[0, sum(additions) * 1.15]
    ),
    waterfallgap=0.3,
    margin=dict(t=80, b=80, l=80, r=40),
)

fig.show()